# EvidenceLab #03: Competing Risks
## When Another Event Gets There First
**Dr. Amobi Andrew Onovo | See it. Understand it. Run it.**

**Define the event → identify what can happen first → choose the estimand → then choose the model.**

Welcome. We will follow 2,500 simulated people from a common starting point, draw event-probability curves, and ask what changes when another event happens first. No upload or patient data is required. Choose **Runtime → Run all** in Google Colab. The first cell installs the analysis library. Allow a few minutes. Outputs are written to the runtime's `evidencelab03_outputs` folder; download that folder before the temporary runtime expires.

### Four ideas before the code

| Idea | Plain-language meaning |
|---|---|
| **Time-to-event data** | We study whether something happens **and how long it takes**. |
| **Event of interest** | The outcome we want to study: for example, relapse, death, resignation or equipment failure. |
| **Censoring** | Observation stops before we see the event. This does not necessarily mean another event made it impossible. |
| **Competing event** | Another event happens first and prevents the chosen outcome from occurring later under our endpoint definition. |

**The key question: What can happen first, and does it change what can happen next?**

An **estimand** is the quantity we want to estimate. Here, one estimand is the probability of first relapse within 60 months, before death. **Time zero** is the point when follow-up starts.

Our teaching example is **first cancer relapse**, with **death before relapse** competing. Time zero is a hypothetical start of follow-up after treatment. These artificial data do not describe real cancer patients, HIV outcomes, or treatment effectiveness.

### The research story
During his PhD, Dr. Onovo analyzed survival among adults starting antiretroviral therapy (ART) in Nigeria. Professor Olivia Keiser asked: “What about the competing risks?” He learned the method, rebuilt the analysis in Stata, and incorporated it into work presented at IAS 2017 in Paris, 23–26 July 2017. Abstract **MOPEB0307**, printed page **79**, documents the historical analysis.

**Important distinction:** the abstract accounted for loss to follow-up using competing-risk regression. Loss to follow-up (LTFU) does not biologically prevent death. For all-cause mortality it may be missing outcome information or censoring, potentially informative—that is, losing contact may be related to the person’s outcome. For a first recorded program outcome, a departure can be a competing endpoint; a multi-state model, which follows transitions between states such as in care, out of care and returned to care, may be preferable for departures and returns. Define the question first.

**Choose your reading path.** On a first visit, run all cells and focus on the plots, interpretations and exercises. Sections marked **Optional technical detail** still run automatically; return to their mathematics and diagnostic plots when you want more depth. A **risk set** means the people still eligible to experience the first event at a particular time. A **covariate** or **predictor** is a measured characteristic included in a model.

## 1. Setting up the tools

A Python **package** is a collection of reusable code. This lesson uses packages that provide tested functions for survival and competing-risk analysis. Google Colab already includes many common data-science packages. We install a specific version of **lifelines**, the package that supplies our survival-analysis functions, so everyone uses the same implementation. The notebook also records other package versions and uses a fixed random seed to support reproducibility.

**What you need to do:** Run the cell below. Colab installs the package automatically. `%pip` is the notebook command that installs packages; `==0.30.3` selects a specific version. You do not need to type an installation command elsewhere or upload any data.

In [ ]:
%pip -q install lifelines==0.30.3

NumPy generates repeatable random draws and works with arrays. pandas organizes tables. Matplotlib draws and exports figures. lifelines supplies Kaplan–Meier, Aalen–Johansen and Cox estimators. Standard-library modules record versions, dates and files. IPython displays lesson outputs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from lifelines import KaplanMeierFitter, AalenJohansenFitter, CoxPHFitter
from lifelines.statistics import proportional_hazard_test
SEED = 201703
OUTPUT = Path("evidencelab03_outputs")
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "white", "axes.titleweight": "bold", "svg.fonttype": "none"})
COLORS = {0: "#687D92", 1: "#C82333", 2: "#008879", "km": "#1261CE"}

## 2. Specify the question and simulation

### Working with simulated data
We generate artificial records so that we know exactly how every event was generated. This lets us change the competing-event process deliberately and observe what happens to the results. These are teaching data, not patient data or clinical findings. **You do not need to upload a dataset.**

A planned study-end date produces **administrative censoring** for people who remain event-free.

Our question is: **What is the probability of first cancer relapse within 60 months, before death?** We follow 2,500 simulated people from a common starting point. Death before relapse is the competing event. Observation can also end early or at the 60-month study limit.

The settings below control cohort size, follow-up and event rates. A **random seed** is a fixed starting setting for the random-number generator; keeping it fixed lets us repeat the same simulated experiment. Run the baseline unchanged first. Later, change only the competing-event multiplier.

In [ ]:
PARAMETERS = {"n": 2500, "horizon_months": 60.0, "primary_rate": 0.009,
              "competing_rate": 0.010, "censor_rate": 0.0015,
              "competing_multiplier": 1.0}
display(pd.Series(PARAMETERS, name="Simulation setting").to_frame())

### Reusable simulation helper
Think of two waiting times for each person: time until relapse and time until death. We record the event that happens first, unless observation stops sooner. The helper below performs that bookkeeping. You can run it without editing it.

<details><summary>Optional technical detail: how the waiting times are generated</summary>

An **exponential waiting time** is a random duration generated using a specified constant rate for that person. A **latent clock** is a potential event time generated before we select the first observed event. **Conditionally independent** means independent after accounting for the included predictors. A **standardized baseline score** is measured in units of its standard deviation. **Frailty** refers to unmeasured differences in underlying event rates.

Each person has age, a simplified binary sex variable, a randomly assigned exposure, and a measured baseline score. We generate two exponential waiting times with covariate-dependent rates and record whichever comes first. Independent random censoring and a 60-month administrative end can stop observation earlier. Exponential waiting times are a teaching device, not a claim that real hazards are constant. Latent clocks are conditionally independent **in this simulation**; the observed-data cumulative incidence function (CIF) estimator does not require independence between hypothetical latent event times.

The baseline score is measured, not an unobserved frailty. All predictors that determine the simulated primary hazard enter the Cox model. Sex coding is deliberately simplified and is not a statement about the diversity of real populations. Reusing the seed keeps participant covariates and primary waiting times identical across competing-event experiments.

</details>

In [ ]:
def simulate_cohort(n=2500, horizon_months=60, primary_rate=0.009,
                    competing_rate=0.010, censor_rate=0.0015,
                    competing_multiplier=1.0, seed=SEED):
    """Return observed first-event data; shared seed enables paired scenarios."""
    rng = np.random.default_rng(seed)
    age = np.clip(rng.normal(50, 12, n), 20, 85)
    sex_male = rng.binomial(1, 0.5, n)
    exposure = rng.binomial(1, 0.5, n)
    baseline_score = rng.normal(0, 1, n)
    rate1 = primary_rate * np.exp(0.20*(age-50)/10 + 0.10*sex_male - 0.30*exposure + 0.35*baseline_score)
    rate2 = competing_rate * competing_multiplier * np.exp(0.35*(age-50)/10 + 0.10*sex_male + 0.15*baseline_score)
    clock1 = rng.exponential(1/rate1)
    clock2 = rng.exponential(1/rate2)
    censor = np.minimum(rng.exponential(1/censor_rate, n), horizon_months)
    followup = np.minimum(np.minimum(clock1, clock2), censor)
    event = np.where(censor <= np.minimum(clock1, clock2), 0, np.where(clock1 < clock2, 1, 2))
    return pd.DataFrame({"age": age, "sex_male": sex_male, "exposure": exposure,
        "baseline_score": baseline_score, "followup_months": followup, "event_type": event})

In [ ]:
data = simulate_cohort(**PARAMETERS)
assert data.shape == (PARAMETERS["n"], 6)
assert set(data.event_type) == {0, 1, 2}
assert data.followup_months.between(0, PARAMETERS["horizon_months"]).all()
data.to_csv(OUTPUT / "simulated_cohort.csv", index=False)
display(data.head().round(2))
print(f"{len(data):,} people; {data.shape[1]} columns. All records are simulated.")

## 3. Meet the data

**What this step does:** Read the dictionary, counts and follow-up summary.

An event code without a clear definition can lead to the wrong analysis.

**What to look for:** Censored people have no observed first event before observation ends.

**Try changing:** Change the administrative horizon and rerun from simulation to see the counts change.

In [ ]:
dictionary = pd.DataFrame([
    ["age", "Age in years at time zero (20–85)"],
    ["sex_male", "Simplified simulated indicator: 1 male, 0 female"],
    ["exposure", "Randomized simulated exposure: 1 yes, 0 no"],
    ["baseline_score", "Measured standardized baseline risk score"],
    ["followup_months", "Time to first event or censoring, in months"],
    ["event_type", "0 censored; 1 relapse; 2 death before relapse"]], columns=["Variable", "Definition"])
display(dictionary)
STATUS = {0: "Censored", 1: "Relapse", 2: "Death before relapse"}
counts = data.event_type.value_counts().reindex([0, 1, 2], fill_value=0)
display(pd.DataFrame({"Status": [STATUS[k] for k in counts.index], "Count": counts.values,
                      "Percent": (100*counts.values/len(data)).round(1)}))
display(data.followup_months.describe().to_frame("Observed follow-up (months)"))

### Reusable figure-export helper
This helper displays each chart and saves it in two formats: **PNG**, a high-resolution image, and **SVG**, a vector graphic that stays sharp when resized. **CSV** files store tables, and **JSON** files store structured numerical results. Run this cell once; later chart cells use the helper automatically.

In [ ]:
def export_figure(fig, name):
    """Save the same plotted figure for notebook, video and repository reuse."""
    fig.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{name}.svg", bbox_inches="tight")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh([STATUS[k] for k in counts.index], counts.values, color=[COLORS[k] for k in counts.index])
ax.bar_label(bars, padding=8, fmt="%d")
ax.set(xlabel="People (simulated)", title="How observation ends", xlim=(0, counts.max()*1.18))
ax.invert_yaxis()
fig.tight_layout()
export_figure(fig, "event_status")

## 4. See time and observation

**What this step does:** Plot follow-up lengths and twelve individual timelines.

Observation ending is different from an event occurring.

**What to look for:** Symbols show the recorded endpoint; the bar ends there. A large mass at 60 months reflects administrative censoring.

**Try changing:** Display a different twelve-person slice; do not infer population patterns from this tiny illustration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(data.followup_months, bins=np.arange(0, 65, 5), color="#1261CE", edgecolor="white")
axes[0].set(xlabel="Observed follow-up (months)", ylabel="People", title="Follow-up is not the same for everyone")
for row, person in enumerate(data.head(12).itertuples()):
    axes[1].hlines(row, 0, person.followup_months, color=COLORS[person.event_type], lw=2)
    axes[1].plot(person.followup_months, row, marker={0:"|",1:"o",2:"X"}[person.event_type], color=COLORS[person.event_type], ms=8)
axes[1].set(xlabel="Months since time zero", ylabel="Illustrative person", title="First recorded event or censoring", xlim=(0, 62))
fig.tight_layout()
export_figure(fig, "followup_and_timelines")

## 5. Kaplan–Meier: name the outcome first

**Kaplan–Meier (KM)** estimates the probability of remaining free of a defined event over time. It is a useful survival-analysis method. The issue here arises when a competing event is coded as ordinary censoring and **1 − KM** is then interpreted as the absolute probability of relapse before death.

Consider two people who have not relapsed by month 12:

| What happens next? | What does it mean for first relapse? |
|---|---|
| The study ends while the person is alive and relapse-free. | We no longer observe them, but a later relapse remains possible: **censoring**. |
| The person dies before relapse. | A future first relapse can no longer occur: **competing event**. |

The cell below fits two KM calculations. The first treats either relapse or death as an event and answers “What proportion have experienced neither event yet?” The second records relapse as the event and codes deaths as censored. We will compare **1 − that second KM curve** with cumulative incidence in the next section.

Read the first graph from left to right: the vertical axis is the estimated probability of remaining free of both events. Its shaded 95% confidence interval shows sampling uncertainty at each time: under the assumptions, intervals constructed this way would contain the true value in about 95% of repeated samples. We return to these pointwise intervals in the next section.

In [ ]:
km_any = KaplanMeierFitter(label="Free of relapse and death").fit(data.followup_months, data.event_type > 0)
km_relapse = KaplanMeierFitter(label="KM: deaths coded as censored").fit(data.followup_months, data.event_type == 1)
fig, ax = plt.subplots(figsize=(11, 5))
km_any.plot_survival_function(ax=ax, ci_show=True, color="#00345C")
ax.set(xlabel="Months since time zero", ylabel="Probability of neither event", title="Event-free survival: neither relapse nor death", ylim=(0, 1))
export_figure(fig, "event_free_survival")

<details><summary>Optional technical detail: censoring and the cause-specific risk set</summary>

**Interpretation:** ordinary censoring requires that those remaining observed represent those censored for the target, possibly conditional on measured variables. For a cause-specific Cox fit (a model for an event rate among people currently event-free), coding competitors as censored is a computational way to remove them from that cause-specific risk set; it does **not** mean we assume they could still experience the first event. Interpreting 1 − relapse-only KM as risk in a hypothetical world without death needs additional assumptions; it is not a causal result here.

</details>

## 6. Cumulative incidence: the probability question

The **cumulative incidence function (CIF)** estimates the probability of a particular first event by a given time, while accounting for competing events. The **Aalen–Johansen estimator** is the calculation we use to obtain these probabilities. For example, relapse CIF at 60 months answers: “What proportion are estimated to relapse by 60 months, before death?”

We calculate one CIF for relapse and another for death before relapse. Each starts at zero and cannot decrease. Their sum is the estimated probability of either first event. Adding the probability of neither event gives one.

**Try changing this:** after the first full run, set `selected_time` below to 24 or 36 months. Rerun this section and the following cells. The displayed interpretation will use your selected time.

In [ ]:
aj_relapse = AalenJohansenFitter(calculate_variance=True, seed=SEED)
aj_death = AalenJohansenFitter(calculate_variance=True, seed=SEED)
aj_relapse.fit(data.followup_months, data.event_type, event_of_interest=1)
aj_death.fit(data.followup_months, data.event_type, event_of_interest=2)
selected_time = 60.0
summary = {"time_months": selected_time, "km_relapse": float(1-km_relapse.predict(selected_time)),
           "cif_relapse": float(aj_relapse.predict(selected_time)), "cif_death": float(aj_death.predict(selected_time)),
           "event_free": float(km_any.predict(selected_time)),
           "at_risk_just_before": int((data.followup_months >= selected_time).sum())}
summary_labels = {
    "time_months": "Time since follow-up began (months)",
    "km_relapse": "1 - KM, with deaths censored (proportion)",
    "cif_relapse": "Relapse before death: CIF (proportion)",
    "cif_death": "Death before relapse: CIF (proportion)",
    "event_free": "Neither event yet (proportion)",
    "at_risk_just_before": "People still observed and event-free just before this time"}
display(pd.Series(summary).rename(index=summary_labels).to_frame("Estimate"))

### Read the comparison chart
The horizontal axis shows months since follow-up began. The vertical axis shows estimated probability: **0.30 means 30%, or about 30 in 100 people**. The dashed blue curve is 1 − KM with deaths censored; red is relapse CIF; green is death-before-relapse CIF. Compare blue and red at the same month. The green curve describes a different event.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
grid = np.linspace(0, 60, 601)
ax.step(grid, 1-km_relapse.predict(grid), where="post", color=COLORS["km"], ls="--", lw=2.5, label="1 − KM (death coded as censored)")
aj_relapse.plot_cumulative_density(ax=ax, color=COLORS[1], ci_show=False, lw=2.8, label="CIF: relapse")
aj_death.plot_cumulative_density(ax=ax, color=COLORS[2], ci_show=False, lw=2.5, ls="-.", label="CIF: death before relapse")
ax.set(xlabel="Months since time zero", ylabel="Estimated probability", xlim=(0, 60), ylim=(0, 0.65), title="Same data. Different answers.")
ax.legend(loc="upper left", frameon=False)
fig.text(0.13, 0.01, f"At {selected_time:g} months: 1 − KM {summary['km_relapse']:.1%} | relapse CIF {summary['cif_relapse']:.1%} | death CIF {summary['cif_death']:.1%}", fontsize=13)
fig.tight_layout(rect=(0, .06, 1, 1))
export_figure(fig, "km_cif_comparison")

In [ ]:
learner_interpretations = {}
learner_interpretations["km_cif"] = (
    f"By {selected_time:g} months, 1 − Kaplan–Meier with deaths coded as censored gives "
    f"approximately **{summary['km_relapse']:.1%}**, while the relapse cumulative incidence "
    f"function estimates approximately **{summary['cif_relapse']:.1%}**. "
    "For the probability of relapse before death, use the CIF. "
    "The KM calculation codes competing deaths as censoring; the CIF explicitly accounts "
    "for death happening first and preventing a later first relapse.")
display(Markdown(learner_interpretations["km_cif"]))
display(Markdown("> **When competing events are present, treating them as ordinary censoring "
                 "can overestimate the absolute probability of the event of interest.**"))

The gap between blue and red reflects different quantities being estimated; it is not a comparison of predictive accuracy between two fitted regression models.

The next graph adds **95% confidence intervals**: shaded ranges expressing sampling uncertainty around each estimated probability. A method with 95% coverage would contain the true value in about 95% of repeated samples under its assumptions. **Pointwise** means this applies separately at a chosen time, not to the entire curve at once. The band is not a prediction interval for one person. The table below it counts people still observed and event-free just before each time; fewer people remaining generally means less information about later follow-up.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
aj_relapse.plot_cumulative_density(ax=ax, color=COLORS[1], label="Relapse CIF", ci_alpha=.15)
aj_death.plot_cumulative_density(ax=ax, color=COLORS[2], label="Death-before-relapse CIF", ci_alpha=.15)
ax.set(title="Cumulative incidence, with pointwise uncertainty", xlabel="Months since time zero", ylabel="Estimated probability", xlim=(0,60), ylim=(0,.65))
export_figure(fig, "cif_uncertainty")
display(pd.DataFrame({"Month": [0,12,24,36,48,60], "Still at risk just before": [(data.followup_months >= t).sum() for t in [0,12,24,36,48,60]]}))

## 7. Cause-specific Cox: a rate question

**A hazard is an instantaneous event rate**, among people still at risk at that point in time. It is not the probability that an event will happen over the next five years. A **hazard ratio (HR)** compares those instantaneous rates between groups, conditional on the fitted model and its included predictors.

**Hazard ratio ≠ risk ratio ≠ probability.** A risk ratio compares probabilities over a specified period; a hazard ratio compares instantaneous rates.

Our question is: **Among individuals who are currently event-free, how is this predictor associated with the instantaneous relapse rate?**

The Cox model below compares simulated exposure groups while accounting for age, sex and baseline score. “Accounting for” means comparing at the same values of the other modeled predictors. In the table, HR = 1 means equal rates; HR below 1 means a lower rate; HR above 1 means a higher rate. These are associations in simulated data, not causal treatment effects.

The 95% confidence interval describes estimation uncertainty. A **p-value** measures how incompatible the data are with a specified null model, subject to its assumptions; it is not the probability that the null hypothesis is true. Here, that null value is HR = 1. Read the estimate and interval together rather than relying on a threshold.

**Try changing this:** compare the age and exposure rows. Age is expressed per **10-year increase**; exposure compares yes with no. Leave the model unchanged for the reference run.

In [ ]:
cox_data = data.assign(age_decades=(data.age-50)/10, relapse=(data.event_type==1).astype(int))[
    ["followup_months", "relapse", "age_decades", "sex_male", "exposure", "baseline_score"]]
cox = CoxPHFitter().fit(cox_data, duration_col="followup_months", event_col="relapse")
cox_table = cox.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
predictor_labels = {
    "age_decades": "Age: per 10-year increase",
    "sex_male": "Simulated sex: male versus female",
    "exposure": "Simulated exposure: yes versus no",
    "baseline_score": "Baseline score: per 1 standard-deviation increase"}
readable_cox_table = cox_table.rename(index=predictor_labels, columns={
    "exp(coef)": "Hazard ratio", "exp(coef) lower 95%": "95% CI: lower",
    "exp(coef) upper 95%": "95% CI: upper", "p": "p-value"})
display(readable_cox_table.round(3))
cox_table.to_csv(OUTPUT / "cause_specific_cox.csv")
hr = float(cox.hazard_ratios_["exposure"])

In [ ]:
exposure_interval = cox_table.loc["exposure", ["exp(coef) lower 95%", "exp(coef) upper 95%"]]
rate_difference_percent = abs(hr - 1) * 100
direction = "lower" if hr < 1 else "higher"
learner_interpretations["cox_statistical"] = (
    f"**Statistical interpretation:** exposure HR = **{hr:.2f}** "
    f"(95% CI **{exposure_interval.iloc[0]:.2f}–{exposure_interval.iloc[1]:.2f}**) "
    "for the cause-specific relapse hazard, conditional on age, sex and baseline score.")
learner_interpretations["cox_plain"] = (
    f"**Plain-language interpretation:** among people still alive and relapse-free, "
    f"the exposed group has an estimated **{rate_difference_percent:.1f}% {direction} "
    "instantaneous relapse rate** than the unexposed group at the same modeled predictor values. "
    "This does not mean the same percentage reduction or increase in five-year relapse probability. "
    "It is a fitted association in this artificial cohort, not evidence of a real treatment effect.")
display(Markdown(learner_interpretations["cox_statistical"]))
display(Markdown(learner_interpretations["cox_plain"]))

## 8. Check the Cox assumptions — optional technical detail

The model assumes that a predictor's hazard ratio is constant over follow-up, even though the underlying event rate can change. This is the **proportional-hazards assumption**.

The plots below use **scaled Schoenfeld residuals**, quantities that compare a predictor's value at an event with what the fitted model would expect. They help us look for changes in a predictor's association over time. You do not need to calculate these by hand.

**How to read the result:** individual dots are noisy. Look for a persistent trend in the red smoothed line, not an isolated point. A trend can challenge a constant hazard ratio. The diagnostic p-value tests for a time pattern; a large value does not prove the model is correct. The smoother is descriptive, not a formal confidence band.

**For further study:** functional form describes how a predictor enters the model, for example linearly or as a curve. A time-varying effect allows its association to change over time. Stratification allows separate baseline rates for specified groups. Choose a remedy using the study context and diagnostics.

### Reusable diagnostic-plot helper
This cell defines the same residual plots used in the validated analysis. The next cell runs the check and displays them.

In [ ]:
def plot_cox_diagnostics(cox, cox_data):
    """Display scaled residuals and a descriptive rolling smoother."""
    residuals = cox.compute_residuals(cox_data, kind="scaled_schoenfeld")
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for ax, name in zip(axes.flat, residuals.columns):
        time = cox_data.loc[residuals.index, "followup_months"]
        ordered = pd.DataFrame({"time": time, "residual": residuals[name]}).sort_values("time")
        ax.scatter(ordered.time, ordered.residual, s=7, alpha=.2, color="#1261CE")
        ax.plot(ordered.time, ordered.residual.rolling(80, center=True, min_periods=30).mean(), color="#C82333", lw=2)
        ax.axhline(0, color="gray", lw=1)
        ax.set(title=name, xlabel="Event time (months)", ylabel="Scaled residual")
    fig.suptitle("Look for time trends; the red smoother is descriptive")
    fig.tight_layout()
    export_figure(fig, "cox_diagnostics")

In [ ]:
ph_test = proportional_hazard_test(cox, cox_data, time_transform="rank")
display(ph_test.summary.rename(index=predictor_labels, columns={"test_statistic": "Diagnostic statistic", "p": "p-value", "-log2(p)": "-log2(p), technical scale"}))
ph_test.summary.to_csv(OUTPUT / "cox_ph_diagnostic.csv")
plot_cox_diagnostics(cox, cox_data)

**Decision:** interpret the diagnostic together with the plots and the study design, not as a pass/fail button. Our conditional simulation uses proportional hazards and linear predictors by construction. A finite sample can still flag a pattern by chance. The analysis requires correct event coding and suitable independent censoring; this test cannot verify those. For real data, investigate functional form, measurement, missingness and confounding (other factors associated with both the predictor and outcome). Cox associations alone do not establish causality.

## 9. Fine–Gray: a different association

**Cause-specific Cox asks:** among those currently event-free, how does a predictor relate to the instantaneous event rate?

**Fine–Gray asks:** how does a predictor relate to the subdistribution hazard underlying the cumulative incidence of the event?

The **subdistribution hazard** is a mathematical quantity used to model associations with a particular CIF. Unlike the cause-specific hazard, it is not the ordinary event rate among only those still event-free. Its ratio is called a **subdistribution hazard ratio (sHR)**. An sHR is not a probability or a risk ratio.

This notebook only executes statistical methods validated in the selected Colab environment. Fine–Gray is explained conceptually rather than implemented through an unverified workaround.

**Different models answer different questions.** Choose the quantity your question requires before choosing a model.

<details><summary>Optional technical detail: Fine–Gray risk sets and implementation</summary>

The selected lifelines 0.30.3 stack provides Kaplan–Meier, Aalen–Johansen and Cox. The established R package [`cmprsk::crr`](https://cran.r-project.org/web/packages/cmprsk/cmprsk.pdf) implements Fine–Gray for further work; using it here would require a separately validated runtime.

Fine–Gray uses a modified mathematical risk set that retains people who experienced a competing event, with censoring weights where needed. These weights account for incomplete observation under the model's censoring assumptions. Keeping someone in that mathematical calculation does not mean they remain physically able to have a first relapse after death.

Under a correctly specified proportional subdistribution hazards model, a coefficient describes an association with the subdistribution hazard and hence the modeled CIF. Predictions require a baseline function (the reference hazard pattern) and predictor values. Check proportional subdistribution hazards and censoring assumptions. Cause-specific models for all causes can also be combined to predict CIFs.

</details>

In [ ]:
comparison = pd.DataFrame([
    ["Kaplan–Meier", "Survival for a defined endpoint", "How long until the endpoint?", "Any-event survival; suitable ordinary censoring"],
    ["Aalen–Johansen CIF", "Absolute probability by event type", "What proportion relapse before death by t?", "Competing-risk probability curves"],
    ["Cause-specific Cox", "Cause-specific hazard ratio", "How do covariates relate to rates while event-free?", "Conditional event-rate associations"],
    ["Fine–Gray (conceptual)", "Subdistribution hazard ratio", "How do covariates relate to the subdistribution hazard?", "CIF-related regression associations"]],
    columns=["Method", "Primary quantity", "Question", "Use"])
display(comparison)
comparison.to_csv(OUTPUT / "method_comparison.csv", index=False)

## 10. The experiment: change what can happen first

**Prediction before computation:** if the relapse mechanism stays the same but death before relapse becomes more common, what do you think will happen to relapse cumulative incidence?

The next cell runs the same low, moderate and high competing-event scenarios. It reuses the same random draws for each person's predictors, relapse waiting time and censoring time. Only the competing-event multiplier changes. This paired comparison helps us see the effect of that one simulation setting.

**Try changing this:** after the reference run, change the high multiplier from `2.5` to `3.0` in the list below. Keep the other settings and seed unchanged. Rerun this cell and the following cells. The fixed scenarios work with ordinary Run all and require no interactive widget setup.

In [ ]:
scenario_curves, scenario_rows = {}, []
fig, ax = plt.subplots(figsize=(12, 6))
for label, multiplier, color, style in [("Low", .25, "#1261CE", "-"), ("Moderate", 1., "#C82333", "--"), ("High", 2.5, "#008879", "-.")]:
    settings = {**PARAMETERS, "competing_multiplier": multiplier}
    cohort = simulate_cohort(**settings)
    estimator = AalenJohansenFitter(calculate_variance=False, seed=SEED).fit(cohort.followup_months, cohort.event_type, event_of_interest=1)
    curve = np.asarray(estimator.predict(grid))
    scenario_curves[label] = curve.tolist()
    scenario_rows.append({"Scenario": label, "Competing multiplier": multiplier, "Relapse CIF at 60 months": float(estimator.predict(60))})
    ax.step(grid, curve, where="post", color=color, ls=style, lw=2.7, label=f"{label} competing-event rate")
ax.set(title="Same relapse mechanism. Different competing-event frequency.", xlabel="Months since time zero", ylabel="Cumulative incidence of relapse", xlim=(0,60), ylim=(0,.65))
ax.legend(frameon=False)
export_figure(fig, "competing_event_scenarios")
scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table)
scenario_table.to_csv(OUTPUT / "scenario_results.csv", index=False)

In [ ]:
scenario_probabilities = scenario_table.set_index("Scenario")["Relapse CIF at 60 months"]
learner_interpretations["scenarios"] = (
    f"At 60 months, relapse CIF is **{scenario_probabilities['Low']:.1%}** in the low "
    f"competing-event scenario, **{scenario_probabilities['Moderate']:.1%}** in the moderate "
    f"scenario and **{scenario_probabilities['High']:.1%}** in the high scenario. "
    "More deaths occur before a possible relapse, so fewer people can experience a first relapse. "
    "The relapse-generating mechanism has stayed fixed. This is a controlled simulation comparison, "
    "not an estimated effect of a real intervention.")
display(Markdown(learner_interpretations["scenarios"]))

## 11. Independent checks — optional technical detail

Reproducible science means checking that important results do not depend blindly on one software function. This section independently verifies key calculations. Run these cells with the rest of the notebook; reading the detailed arithmetic is optional on your first pass.

The helper recomputes cumulative incidence directly from the numbers of people at risk and events at each time. It checks those results against Aalen–Johansen and checks a small hand-calculated example.

Here **S** is the probability of neither event, **CIF1** is relapse probability and **CIF2** is death-before-relapse probability. Their sum must be one. **Monotone** means the cumulative probabilities cannot decrease. If all assertions (automatic checks) succeed, the cell prints PASS. An assertion failure means a check needs investigation, not that you should delete the check.

### Reusable independent arithmetic helper
At each observed time, count those at risk just before it (Y), first relapses (d1), and competing deaths (d2). Add S(previous) × dk/Y to CIFk, then multiply event-free survival by 1 − (d1+d2)/Y. Censoring removes people from later risk sets without adding an event. Events are processed before censoring at a tied time. The library remains the production estimator; this independently written helper checks its arithmetic, including ties.

In [ ]:
def risk_set_recursion(times, events):
    """Transparent Aalen–Johansen recursion with grouped tied times."""
    times, events = np.asarray(times), np.asarray(events)
    survival, cif1, cif2, rows = 1.0, 0.0, 0.0, []
    for time in np.unique(times):
        at_risk = np.sum(times >= time)
        d1 = np.sum((times == time) & (events == 1))
        d2 = np.sum((times == time) & (events == 2))
        cif1 += survival*d1/at_risk
        cif2 += survival*d2/at_risk
        survival *= 1-(d1+d2)/at_risk
        rows.append([time, survival, cif1, cif2])
    return np.asarray(rows)

In [ ]:
audit = risk_set_recursion(data.followup_months, data.event_type)
np.testing.assert_allclose(audit[:,2], aj_relapse.predict(audit[:,0]), atol=1e-10)
np.testing.assert_allclose(audit[:,3], aj_death.predict(audit[:,0]), atol=1e-10)
np.testing.assert_allclose(audit[:,1]+audit[:,2]+audit[:,3], 1, atol=1e-10)
assert np.all(np.diff(audit[:,2:], axis=0) >= -1e-12)
assert np.all(1-np.asarray(km_relapse.predict(audit[:,0])) >= audit[:,2]-1e-12)
toy = risk_set_recursion([1, 2, 3, 4], [1, 2, 0, 1])
np.testing.assert_allclose(toy[-1,1:], [0, .75, .25])
print("PASS: independent risk-set arithmetic, probability identity, monotonicity, KM/CIF ordering and hand example.")

## 12. Cross-sector practice

Competing-risk analysis concerns **time and mutually exclusive first-event pathways**. It is not specific to a disease. Mutually exclusive means a person's recorded first event belongs to one category at that time. State the research question before choosing those categories.

| Setting and research question | Event of interest | Possible competing event | What it prevents |
|---|---|---|---|
| **Health:** How long until first relapse after cancer treatment? | First relapse | Death before relapse | A first relapse cannot occur after death. |
| **Employment:** How long until the first exit from this job is a resignation? | Resignation | Retirement or dismissal first | The person's first exit from this job can no longer be resignation. |
| **Customers:** How long until the first account closure is attributed to price? | First closure attributed to price | First closure attributed to another exclusive reason | That account's first closure cannot subsequently be attributed to price. |
| **Engineering:** How long until a component first fails through degradation? | First degradation failure | Catastrophic failure first | The component cannot later have degradation as its first failure. |

Re-employment, reopened accounts and repairs create additional states. A multi-state or recurrent-event model may be needed when later transitions or repeated events are part of the question. Ambiguous or multiple closure reasons require a clearer endpoint definition.

### Your turn
We study first cancer relapse after treatment. Person A dies without relapse at month 8. Person B is alive and relapse-free when the study ends at month 12. Person C relapses at month 10. Classify each record as event of interest, competing event or censoring. Would losing contact with Person B prove that they could no longer relapse?

<details><summary>Reveal the answer</summary>

**A:** competing event—death before relapse. **B:** censored—observation ends without either recorded event. **C:** event of interest—first relapse. Losing contact does not make relapse impossible; it leaves the later outcome unknown. If loss of contact is related to outcome risk, ordinary censoring assumptions may not be adequate.

For the other rows, remaining employed, keeping the account active or having a functioning component when observation ends corresponds to censoring. Replace one row with your own question and identify the three categories.

</details>

## 13. Decision guide

**Is your outcome time to an event?**  
↓ Define time zero, eligibility and follow-up. If not, use a method suited to your outcome.  
**Can another event happen first?**  
↓ If yes, identify it.  
**Would it prevent your event of interest from occurring later under the chosen definition?**  
↓ If yes, treat it as a competing event. If observation merely stops, consider censoring and its assumptions.  
**What do you want to estimate?**

| Quantity | Method to consider |
|---|---|
| Absolute event probability by event type | CIF, estimated here with Aalen–Johansen |
| Instantaneous event rate among people currently event-free | Cause-specific hazard; Cox can model predictor associations |
| Covariate association with the CIF through a subdistribution hazard | Fine–Gray |
| Probability of neither event yet | Any-event survival, for example with KM |

**The research question and estimand come before the model.**

For mortality after loss to follow-up, ask what vital-status information is missing. Linkage (connecting records), tracing (finding later outcomes), sensitivity analysis (examining alternative assumptions) or multi-state modeling may be needed. Changing an event code cannot recover an unknown outcome.

## 14. Save the analysis and results

The final code cell saves the simulated records, figures, full-precision estimates, chart coordinates, generated interpretations and package versions. A **claim ledger** is a table connecting numerical statements to their source. Metadata records how the analysis was run.

To keep your work, open Colab's Files panel, find `evidencelab03_outputs`, and download the files you need before the temporary runtime expires. You can rerun the notebook to regenerate them. **Try changing this:** compare outputs from two fresh runs with the same settings; estimates should agree apart from tiny numerical rounding differences.

In [ ]:
summary["exposure_cause_specific_hr"] = hr
summary["n"] = len(data)
metadata = {"seed": SEED, "parameters": PARAMETERS, "generated_utc": datetime.now(timezone.utc).isoformat(),
            "packages": {name: version(name) for name in ["numpy", "pandas", "scipy", "matplotlib", "lifelines"]},
            "fine_gray": "conceptual only; no fitted sHR", "data": "entirely simulated"}
curves = {"months": grid.tolist(), "km_relapse": (1-np.asarray(km_relapse.predict(grid))).tolist(),
          "cif_relapse": np.asarray(aj_relapse.predict(grid)).tolist(), "cif_death": np.asarray(aj_death.predict(grid)).tolist(),
          "scenarios": scenario_curves}
for name, value in [("results", summary), ("metadata", metadata), ("chart_data", curves)]:
    (OUTPUT / f"{name}.json").write_text(json.dumps(value, indent=2), encoding="utf-8")
ledger = pd.DataFrame([{"claim": key, "full_value": value, "source": "Notebook summary dictionary", "scope": "Simulated cohort; selected time where applicable"} for key, value in summary.items()])
ledger.to_csv(OUTPUT / "claim_ledger.csv", index=False)
display(metadata)
print("EVIDENCELAB03_RUN_ALL_PASS")
(OUTPUT / "learner_interpretations.json").write_text(json.dumps(learner_interpretations, indent=2), encoding="utf-8")
print("EVIDENCELAB03_RESULTS_JSON=" + json.dumps(summary))
print("EVIDENCELAB03_SCENARIOS_JSON=" + scenario_table.to_json(orient="records"))
print("EVIDENCELAB03_METADATA_JSON=" + json.dumps(metadata))

## 15. Quick reference

| Term | Meaning |
|---|---|
| **KM** | Kaplan–Meier; estimates survival for a defined endpoint. |
| **CIF** | Cumulative incidence function; probability of a particular first event by a specified time. |
| **HR** | Hazard ratio; a comparison of instantaneous event rates, conditional on the model. |
| **Cause-specific hazard** | Instantaneous rate of a particular event among those currently event-free. |
| **Subdistribution hazard** | Quantity used by Fine–Gray regression to model covariate associations with cumulative incidence. |
| **sHR** | Subdistribution hazard ratio; neither a probability nor a risk ratio. |
| **Competing event** | An event that happens first and prevents the event of interest from occurring later under the defined endpoint. |
| **Estimand** | The quantity the research question asks us to estimate. |

### If you remember only one thing
When another event can happen first, do not automatically treat it as ordinary censoring. Ask what that event means for your scientific question.

**The EvidenceLab workflow:**  
**QUESTION → TIME → EVENT → COMPETITOR → ESTIMATE → MODEL → INTERPRET**

**Define the event. Identify what can happen first. Choose the estimand. Then choose the model.**

**Different models answer different questions.**

Try one experiment: increase the competing-event multiplier, rerun the notebook, and explain why relapse CIF changes even though the relapse-generating mechanism is unchanged.

### Sources and further study
* [IAS 2017 abstract book, MOPEB0307, printed page 79](https://www.ias2017.org/Portals/1/Files/IAS2017_LO.compressed4c6a.pdf?fileticket=m3LSDs1z4QY%3d&tabid=577&portalid=1).
* [lifelines Aalen–Johansen documentation](https://lifelines.readthedocs.io/en/latest/fitters/univariate/AalenJohansenFitter.html). Continuous event times avoid event ties here; administrative censoring ties remain valid.
* [lifelines Cox regression](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html) and [proportional hazards diagnostics](https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html).
* [scikit-survival competing-risk estimator](https://scikit-survival.readthedocs.io/en/stable/api/generated/sksurv.nonparametric.cumulative_incidence_competing_risks.html), used in external independent validation.
* Fine JP, Gray RJ (1999). [A proportional hazards model for the subdistribution of a competing risk](https://doi.org/10.1080/01621459.1999.10474144). JASA 94:496–509.
* [CRAN cmprsk manual](https://cran.r-project.org/web/packages/cmprsk/cmprsk.pdf), validated R implementation for further study.

Educational simulation, not clinical advice or patient evidence. **EvidenceLab | See it. Understand it. Run it.**